# Marker-Aligned Subspace Separation for Implicit Sentiment Analysis (MASS)
**Google Colab Runner — Supports both `Flan-T5-Base` (Full Fine-Tuning) & `Flan-T5-XXL` (LoRA, `bfloat16`), plus Table 7 Ablations**

This notebook can be run directly on Google Colab (or locally in Jupyter) in two ways:
1. **Self-Contained Notebook Mode** (Cells 1–5): Run the complete MASS pipeline directly inside this notebook for `Flan-T5-Base`, `Flan-T5-XXL`, and all 7 Table 7 ablation variants.
2. **CLI Mode** (Cell 6): Run `main.py` from the repository directory.

In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP (Colab / Local)
# ==============================================================================
!pip install -q transformers peft accelerate tqdm numpy scikit-learn sentencepiece PyYAML

import os

# Optional: Mount Google Drive when running on Google Colab
USE_GOOGLE_DRIVE = False  # Set to True if your data/outputs are stored on Google Drive
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/MASS"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = DRIVE_PROJECT_DIR
else:
    PROJECT_DIR = "." if os.path.isdir("./data") else "./MASS"

print(f"Working project directory: {os.path.abspath(PROJECT_DIR)}")

In [ ]:
# ==============================================================================
# CELL 2: CORE MASS IMPLEMENTATION (Flan-T5-Base & Flan-T5-XXL + Table 7 Ablations)
# ==============================================================================
import csv
import json
import os
import random
import xml.etree.ElementTree as ET
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import T5EncoderModel, T5TokenizerFast, get_linear_schedule_with_warmup

POLARITY_TO_NUM = {"positive": 0, "negative": 1, "neutral": 2}
NUM_TO_POLARITY = {0: "positive", 1: "negative", 2: "neutral"}

MODEL_PRESETS = {
    "base": {
        "pretrained_name": "google/flan-t5-base",
        "dtype": "float32",
        "use_lora": False,
        "lora": None,
    },
    "xxl": {
        "pretrained_name": "google/flan-t5-xxl",
        "dtype": "bfloat16",
        "use_lora": True,
        "lora": {
            "rank": 16,
            "alpha": 32,
            "dropout": 0.05,
            "target_modules": ["q", "v"],
        },
    },
}

ABLATION_PRESETS = {
    "full": {
        "use_aspect_marker": True,
        "representation_mode": "start_marker",
        "use_two_subspaces": True,
        "use_orthogonality": True,
        "use_class_weights": True,
    },
    "wo_aspect_marker": {
        "use_aspect_marker": False,
        "representation_mode": "mean_prompt",
        "use_two_subspaces": True,
        "use_orthogonality": True,
        "use_class_weights": True,
    },
    "wo_orthogonality": {
        "use_aspect_marker": True,
        "representation_mode": "start_marker",
        "use_two_subspaces": True,
        "use_orthogonality": False,
        "use_class_weights": True,
    },
    "wo_two_subspaces": {
        "use_aspect_marker": True,
        "representation_mode": "start_marker",
        "use_two_subspaces": False,
        "use_orthogonality": False,
        "use_class_weights": False,
    },
    "wo_class_weights": {
        "use_aspect_marker": True,
        "representation_mode": "start_marker",
        "use_two_subspaces": True,
        "use_orthogonality": True,
        "use_class_weights": False,
    },
    "end_marker": {
        "use_aspect_marker": True,
        "representation_mode": "end_marker",
        "use_two_subspaces": True,
        "use_orthogonality": True,
        "use_class_weights": True,
    },
    "mean_span_pooling": {
        "use_aspect_marker": True,
        "representation_mode": "mean_span",
        "use_two_subspaces": True,
        "use_orthogonality": True,
        "use_class_weights": True,
    },
}


def log(stage, message):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] [{stage}] {message}")


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_prompt(term, marked_text):
    return (
        f"Task: Analyze sentiment for the aspect [{term}]. "
        "Step 1: Is the context stating an objective fact (Neutral) or expressing a subjective opinion (Polar)? "
        "Step 2: If subjective, is it Positive or Negative? "
        f"Context: {marked_text}"
    )


def insert_aspect_markers(text, term, start_attr, end_attr, start_marker="<extra_id_0>", end_marker="<extra_id_1>"):
    try:
        s_idx, e_idx = int(start_attr), int(end_attr)
    except (TypeError, ValueError):
        s_idx = e_idx = None
    if s_idx is not None and e_idx is not None and 0 <= s_idx <= e_idx <= len(text) and text[s_idx:e_idx] == term:
        return f"{text[:s_idx]}{start_marker} {term} {end_marker}{text[e_idx:]}"
    pos = text.find(term)
    if pos != -1:
        return f"{text[:pos]}{start_marker} {term} {end_marker}{text[pos + len(term):]}"
    return f"{start_marker} {term} {end_marker} {text}"


def parse_xml_to_data(xml_path, use_aspect_marker=True):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    samples = []
    for sentence in root.findall("sentence"):
        text_node = sentence.find("text")
        aspect_terms_node = sentence.find("aspectTerms")
        if text_node is None or text_node.text is None or aspect_terms_node is None:
            continue
        text = text_node.text
        for aspect in aspect_terms_node.findall("aspectTerm"):
            term = (aspect.get("term") or "").strip()
            polarity = (aspect.get("polarity") or "").strip().lower()
            if not term or polarity not in POLARITY_TO_NUM:
                continue
            context_text = insert_aspect_markers(text, term, aspect.get("from"), aspect.get("to")) if use_aspect_marker else text
            samples.append({
                "prompt": build_prompt(term, context_text),
                "numeric_label": POLARITY_TO_NUM[polarity],
                "is_implicit": (aspect.get("implicit_sentiment") or "").strip().lower() == "true",
                "raw_text": text,
                "raw_aspect": term,
            })
    return samples


class ABSADataset(Dataset):
    def __init__(self, data, tokenizer, max_source_length=128, use_aspect_marker=True, representation_mode="start_marker"):
        self.data = data
        self.tokenizer = tokenizer
        self.max_source_length = max_source_length
        self.use_aspect_marker = use_aspect_marker
        self.representation_mode = representation_mode
        self.start_marker_id = tokenizer.convert_tokens_to_ids("<extra_id_0>")
        self.end_marker_id = tokenizer.convert_tokens_to_ids("<extra_id_1>")
        self.pad_token_id = tokenizer.pad_token_id
        self.eos_token_id = tokenizer.eos_token_id

    def __len__(self):
        return len(self.data)

    def _truncate_preserving_markers(self, token_ids):
        if len(token_ids) <= self.max_source_length:
            return token_ids
        has_eos = bool(token_ids and token_ids[-1] == self.eos_token_id)
        core_ids = token_ids[:-1] if has_eos else list(token_ids)
        budget = self.max_source_length - (1 if has_eos else 0)
        if not self.use_aspect_marker:
            kept = core_ids[:budget]
            return kept + ([self.eos_token_id] if has_eos else [])
        start_positions = [i for i, t in enumerate(core_ids) if t == self.start_marker_id]
        end_positions = [i for i, t in enumerate(core_ids) if t == self.end_marker_id]
        if start_positions:
            span_start = start_positions[0]
            span_end = end_positions[0] if end_positions and end_positions[0] >= span_start else span_start
            center = (span_start + span_end) // 2
            w_start = max(0, min(center - budget // 2, len(core_ids) - budget))
            w_end = w_start + budget
            if span_start < w_start:
                w_start, w_end = span_start, min(len(core_ids), span_start + budget)
            elif span_end >= w_end:
                w_end = min(len(core_ids), span_end + 1)
                w_start = max(0, w_end - budget)
            kept = core_ids[w_start:w_end]
        else:
            kept = core_ids[:budget]
        return kept + ([self.eos_token_id] if has_eos else [])

    def _build_representation_mask(self, input_ids, attention_mask):
        mask = torch.zeros_like(attention_mask, dtype=torch.float32)
        if not self.use_aspect_marker or self.representation_mode == "mean_prompt":
            mask = attention_mask.float()
        else:
            start_idx = (input_ids == self.start_marker_id).nonzero(as_tuple=True)[0]
            end_idx = (input_ids == self.end_marker_id).nonzero(as_tuple=True)[0]
            if self.representation_mode == "start_marker" and len(start_idx) > 0:
                mask[start_idx[0]] = 1.0
            elif self.representation_mode == "end_marker" and len(end_idx) > 0:
                mask[end_idx[0]] = 1.0
            elif self.representation_mode == "mean_span" and len(start_idx) > 0:
                s = int(start_idx[0].item())
                e = int(end_idx[0].item()) if len(end_idx) > 0 and int(end_idx[0].item()) >= s else s
                mask[s : e + 1] = 1.0
            elif len(start_idx) > 0:
                mask[start_idx[0]] = 1.0
            else:
                mask = attention_mask.float()
        if mask.sum().item() == 0.0:
            mask[0] = 1.0
        return mask

    def __getitem__(self, idx):
        item = self.data[idx]
        raw_ids = self.tokenizer.encode(item["prompt"], add_special_tokens=True, truncation=False)
        clipped = self._truncate_preserving_markers(raw_ids)
        attn = [1] * len(clipped)
        pad_len = self.max_source_length - len(clipped)
        if pad_len > 0:
            clipped = clipped + [self.pad_token_id] * pad_len
            attn = attn + [0] * pad_len
        input_ids = torch.tensor(clipped, dtype=torch.long)
        attention_mask = torch.tensor(attn, dtype=torch.long)
        rep_mask = self._build_representation_mask(input_ids, attention_mask)
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "representation_mask": rep_mask,
            "numeric_label": torch.tensor(item["numeric_label"], dtype=torch.long),
            "is_implicit": torch.tensor(item["is_implicit"], dtype=torch.bool),
            "raw_text": item["raw_text"],
            "raw_aspect": item["raw_aspect"],
        }


class MassModel(nn.Module):
    def __init__(self, model_preset, ablation_cfg, dataset_name, proj_dim=256, dropout=0.2, label_smoothing=0.1, ortho_weight=0.5):
        super().__init__()
        self.use_lora = model_preset["use_lora"]
        self.use_two_subspaces = ablation_cfg["use_two_subspaces"]
        self.use_orthogonality = ablation_cfg["use_orthogonality"]
        self.use_class_weights = ablation_cfg["use_class_weights"]
        self.ortho_weight = ortho_weight

        dtype = torch.bfloat16 if model_preset["dtype"] == "bfloat16" else torch.float32
        encoder = T5EncoderModel.from_pretrained(model_preset["pretrained_name"], torch_dtype=dtype)
        if self.use_lora:
            from peft import LoraConfig, TaskType, get_peft_model
            lora_cfg = model_preset["lora"]
            peft_config = LoraConfig(
                task_type=TaskType.FEATURE_EXTRACTION,
                r=lora_cfg["rank"],
                lora_alpha=lora_cfg["alpha"],
                lora_dropout=lora_cfg["dropout"],
                target_modules=list(lora_cfg["target_modules"]),
            )
            encoder = get_peft_model(encoder, peft_config)
            encoder.print_trainable_parameters()
        self.encoder = encoder
        hidden_dim = self.encoder.config.d_model

        if self.use_two_subspaces:
            self.proj_subj = nn.Sequential(nn.Linear(hidden_dim, proj_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(proj_dim, proj_dim))
            self.proj_pol = nn.Sequential(nn.Linear(hidden_dim, proj_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(proj_dim, proj_dim))
            self.classifier_subj = nn.Linear(proj_dim, 2)
            self.classifier_pol = nn.Linear(proj_dim, 2)

            subj_w = torch.tensor([2.5, 0.6], dtype=torch.float32) if self.use_class_weights else None
            pol_w_tuple = (1.0, 2.5) if dataset_name == "restaurants" else (1.0, 1.0)
            pol_w = torch.tensor(pol_w_tuple, dtype=torch.float32) if self.use_class_weights else None
            self.loss_fn_subj = nn.CrossEntropyLoss(weight=subj_w, label_smoothing=label_smoothing)
            self.loss_fn_pol = nn.CrossEntropyLoss(weight=pol_w, label_smoothing=label_smoothing, reduction="none")
        else:
            self.proj_main = nn.Sequential(nn.Linear(hidden_dim, proj_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(proj_dim, proj_dim))
            self.classifier_main = nn.Linear(proj_dim, 3)
            self.loss_fn_main = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    @staticmethod
    def _pool(hidden_states, representation_mask):
        mask = representation_mask.to(dtype=hidden_states.dtype).unsqueeze(-1)
        denom = mask.sum(dim=1).clamp_min(1.0)
        return ((hidden_states * mask).sum(dim=1) / denom).float()

    def forward(self, input_ids, attention_mask, representation_mask, labels=None, conditional_polarity=False):
        enc_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        aspect_states = self._pool(enc_out.last_hidden_state, representation_mask)

        if not self.use_two_subspaces:
            logits_main = self.classifier_main(self.proj_main(aspect_states)).float()
            loss = self.loss_fn_main(logits_main, labels) if labels is not None else None
            return {"loss": loss, "logits_main": logits_main, "logits_subj": None, "logits_pol": None}

        logits_subj = self.classifier_subj(self.proj_subj(aspect_states)).float()
        if labels is None and conditional_polarity:
            logits_pol = torch.zeros(input_ids.shape[0], 2, device=logits_subj.device, dtype=logits_subj.dtype)
            polar_mask = torch.argmax(logits_subj, dim=1) == 1
            if polar_mask.any():
                logits_pol[polar_mask] = self.classifier_pol(self.proj_pol(aspect_states[polar_mask])).float()
        else:
            logits_pol = self.classifier_pol(self.proj_pol(aspect_states)).float()

        loss = None
        if labels is not None:
            subj_labels = (labels != 2).long()
            loss_subj = self.loss_fn_subj(logits_subj, subj_labels)
            polar_mask = labels != 2
            loss_pol = self.loss_fn_pol(logits_pol[polar_mask], labels[polar_mask]).mean() if polar_mask.any() else logits_pol.sum() * 0.0
            if self.use_orthogonality and self.ortho_weight > 0.0:
                w_s = F.normalize(self.proj_subj[0].weight.float(), p=2, dim=1)
                w_p = F.normalize(self.proj_pol[0].weight.float(), p=2, dim=1)
                loss_ortho = torch.linalg.matrix_norm(w_s @ w_p.T, ord="fro")
                loss = loss_subj + loss_pol + self.ortho_weight * loss_ortho
            else:
                loss = loss_subj + loss_pol
        return {"loss": loss, "logits_subj": logits_subj, "logits_pol": logits_pol, "logits_main": None}

    def backbone_parameters(self):
        if self.use_lora:
            return [p for p in self.encoder.parameters() if p.requires_grad]
        return list(self.encoder.parameters())

    def head_parameters(self):
        modules = [self.proj_main, self.classifier_main] if not self.use_two_subspaces else [self.proj_subj, self.proj_pol, self.classifier_subj, self.classifier_pol]
        return [p for m in modules for p in m.parameters()]


def decode_predictions(outputs):
    if outputs["logits_main"] is not None:
        return torch.argmax(outputs["logits_main"], dim=1)
    p_subj = torch.argmax(outputs["logits_subj"], dim=1)
    p_pol = torch.argmax(outputs["logits_pol"], dim=1)
    return torch.where(p_subj == 0, torch.full_like(p_pol, 2), p_pol)


def evaluate_model(model, dataloader, device):
    model.eval()
    all_preds, all_trues, all_imp, records = [], [], [], []
    with torch.no_grad():
        for batch in dataloader:
            outputs = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                batch["representation_mask"].to(device),
                conditional_polarity=True,
            )
            preds = decode_predictions(outputs).cpu().numpy()
            trues = batch["numeric_label"].numpy()
            imps = batch["is_implicit"].numpy()
            for i in range(len(preds)):
                all_preds.append(int(preds[i]))
                all_trues.append(int(trues[i]))
                all_imp.append(bool(imps[i]))
                records.append({
                    "text": batch["raw_text"][i],
                    "aspect": batch["raw_aspect"][i],
                    "ground_truth": NUM_TO_POLARITY[int(trues[i])],
                    "prediction": NUM_TO_POLARITY[int(preds[i])],
                    "is_implicit": int(bool(imps[i])),
                })
    preds_np, trues_np, imp_np = np.array(all_preds), np.array(all_trues), np.array(all_imp)
    metrics = {
        "all_accuracy": float(accuracy_score(trues_np, preds_np)),
        "all_macro_f1": float(f1_score(trues_np, preds_np, average="macro", zero_division=0)),
        "isa_accuracy": float(accuracy_score(trues_np[imp_np], preds_np[imp_np])),
        "isa_macro_f1": float(f1_score(trues_np[imp_np], preds_np[imp_np], average="macro", zero_division=0)),
    }
    return metrics, records


def run_experiment(dataset="restaurants", model_variant="base", ablation="full", seed=42, epochs=8, batch_size=16):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model_preset = MODEL_PRESETS[model_variant]
    ablation_cfg = ABLATION_PRESETS[ablation]

    xml_map = {
        "laptops": (
            os.path.join(PROJECT_DIR, "data/laptops/Laptops_Train_v2_Implicit_Labeled.xml"),
            os.path.join(PROJECT_DIR, "data/laptops/Laptops_Test_Gold_Implicit_Labeled.xml"),
        ),
        "restaurants": (
            os.path.join(PROJECT_DIR, "data/restaurants/Restaurants_Train_v2_Implicit_Labeled.xml"),
            os.path.join(PROJECT_DIR, "data/restaurants/Restaurants_Test_Gold_Implicit_Labeled.xml"),
        ),
    }
    train_xml, test_xml = xml_map[dataset]
    out_dir = os.path.join(PROJECT_DIR, "outputs", dataset, model_variant, ablation, f"seed_{seed}")
    os.makedirs(out_dir, exist_ok=True)

    log("CONFIG", f"Dataset={dataset} | Model=MASS-{model_variant.upper()} | Ablation={ablation} | Seed={seed} | Device={device}")
    tokenizer = T5TokenizerFast.from_pretrained(model_preset["pretrained_name"])
    train_data = parse_xml_to_data(train_xml, use_aspect_marker=ablation_cfg["use_aspect_marker"])
    test_data = parse_xml_to_data(test_xml, use_aspect_marker=ablation_cfg["use_aspect_marker"])

    train_loader = DataLoader(
        ABSADataset(train_data, tokenizer, 128, ablation_cfg["use_aspect_marker"], ablation_cfg["representation_mode"]),
        batch_size=batch_size,
        shuffle=True,
    )
    test_loader = DataLoader(
        ABSADataset(test_data, tokenizer, 128, ablation_cfg["use_aspect_marker"], ablation_cfg["representation_mode"]),
        batch_size=batch_size * 2,
        shuffle=False,
    )

    model = MassModel(model_preset, ablation_cfg, dataset).to(device)
    optimizer = AdamW(
        [
            {"params": model.backbone_parameters(), "lr": 2e-5},
            {"params": model.head_parameters(), "lr": 5e-4},
        ],
        weight_decay=0.01,
    )
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)

    best_isa_f1 = float("-inf")
    best_metrics = None
    best_records = None
    ckpt_path = os.path.join(out_dir, "best_model.pt")

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}"):
            optimizer.zero_grad(set_to_none=True)
            outputs = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                batch["representation_mask"].to(device),
                batch["numeric_label"].to(device),
            )
            loss = outputs["loss"]
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

        metrics, records = evaluate_model(model, test_loader, device)
        log(
            "EVAL",
            f"Epoch {epoch:02d} | Loss={epoch_loss/len(train_loader):.4f} | "
            f"All Acc={metrics['all_accuracy']*100:.2f}% F1={metrics['all_macro_f1']*100:.2f}% | "
            f"ISA Acc={metrics['isa_accuracy']*100:.2f}% F1={metrics['isa_macro_f1']*100:.2f}%",
        )
        if metrics["isa_macro_f1"] > best_isa_f1:
            best_isa_f1 = metrics["isa_macro_f1"]
            best_metrics = {"dataset": dataset, "variant": model_variant, "ablation": ablation, "seed": seed, "best_epoch": epoch, **metrics}
            best_records = records
            state = {k: v.detach().cpu() for k, v in model.named_parameters() if v.requires_grad} if model.use_lora else model.state_dict()
            torch.save(state, ckpt_path)

    with open(os.path.join(out_dir, "best_metrics.json"), "w", encoding="utf-8") as f:
        json.dump(best_metrics, f, indent=2)
    for fn, rows in [("full_test_predictions.csv", best_records), ("isa_test_predictions.csv", [r for r in best_records if r["is_implicit"] == 1])]:
        with open(os.path.join(out_dir, fn), "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["text", "aspect", "ground_truth", "prediction", "is_implicit"])
            writer.writeheader()
            writer.writerows(rows)
    log("RESULT", f"Best Epoch={best_metrics['best_epoch']} | ISA F1={best_metrics['isa_macro_f1']*100:.2f}% | Saved to {out_dir}")
    return best_metrics

log("READY", "Loaded MASS pipeline (supports model_variant='base' or 'xxl', and all 7 Table 7 ablations).")

In [ ]:
# ==============================================================================
# CELL 3: RUN MASS WITH FLAN-T5-BASE (Table 4 — Full Fine-Tuning)
# ==============================================================================
run_experiment(dataset="restaurants", model_variant="base", ablation="full", seed=42)
run_experiment(dataset="laptops", model_variant="base", ablation="full", seed=42)

In [ ]:
# ==============================================================================
# CELL 4: RUN MASS WITH FLAN-T5-XXL (Table 4 — bfloat16 + LoRA r=16, alpha=32)
# ==============================================================================
# Note: Requires a GPU with sufficient VRAM (e.g., Colab A100 / L4)
run_experiment(dataset="restaurants", model_variant="xxl", ablation="full", seed=42)
run_experiment(dataset="laptops", model_variant="xxl", ablation="full", seed=42)

In [ ]:
# ==============================================================================
# CELL 5: RUN TABLE 7 ABLATION STUDY & MARKER-REPRESENTATION VARIANTS
# ==============================================================================
ablation_variants = [
    "wo_aspect_marker",
    "wo_orthogonality",
    "wo_two_subspaces",
    "wo_class_weights",
    "end_marker",
    "mean_span_pooling",
]
for ds in ["restaurants", "laptops"]:
    for abl in ablation_variants:
        run_experiment(dataset=ds, model_variant="base", ablation=abl, seed=42)

In [ ]:
# ==============================================================================
# CELL 6: CLI ALTERNATIVE (Run main.py directly across all 3 paper seeds)
# ==============================================================================
# !python main.py --variant base --dataset restaurants laptops --seed 13 42 2024
# !python main.py --variant xxl --dataset restaurants laptops --seed 13 42 2024